# Tech COE Allocation Summary — Gold Analytics

Leadership-facing metrics built on `arganoadw_oacuser_sh.oacuser.TECH_COE_ALLOCATIONS` — the **bronze** fact produced by `tech_coe_allocation_bronze.ipynb` (scheduled hours per **Arganaut × project × week**).

**Pipeline position:** `Excel → [bronze] → TECH_COE_ALLOCATIONS → [this gold notebook] → 6 analytics tables → OAC dashboard`.

**Capacity assumption:** `40` hrs/week per Arganaut (edit `CAPACITY_HOURS_PER_WEEK`).

## Hour types (`HOURS_TYPE`)
Holidays and time off are booked on the project line, so each row is classified:

| HOURS_TYPE | Match | Counts as |
|---|---|---|
| `TIME_OFF` | PROJECT_NAME = *Time Off Bookings* (code `ARG*`) | Non-billable time off |
| `HOLIDAY` | *Holiday - Holiday* | Non-billable time off |
| `BILLABLE` | everything else (client projects) | Billable allocation |

## Key formulas
- **Billable hours** = scheduled hours on `BILLABLE` lines.
- **Available capacity** = `40 − time off` (hours a person can actually work that week).
- **Non-billable / bench** = `max(40 − billable − time off, 0)` (truly idle, sellable capacity).
- **Gross utilization** = `billable ÷ 40`.   **Net utilization** = `billable ÷ available capacity` (so a holiday week isn't penalized). Both are reported.
- Time Off / Holiday are **excluded** from project demand & key-person-risk rankings.

## Gold tables produced
| Table | Grain | Use |
|---|---|---|
| `TECH_COE_ALLOC_DETAIL` | dept × arganaut × project × week | Flexible base (carries `HOURS_TYPE`) — **slice by Department and Project** |
| `TECH_COE_ALLOC_UTILIZATION_PERSON_WEEK` | arganaut × week | Per-person gross/net util, time off, over-allocation, bench |
| `TECH_COE_ALLOC_DEPT_WEEK` | dept × week | Gross & net util %, billable / time-off / bench hours |
| `TECH_COE_ALLOC_PROJECT_WEEK` | project × week | Billable project demand over time |
| `TECH_COE_ALLOC_PROJECT_RISK` | project | Billable total hours, headcount, key-person-risk flag |
| `TECH_COE_ALLOC_DEPT_FORECAST` | dept × week (+ future) | Billable demand trend + moving-average projection |

**Forecast:** linear trend + 3-week moving average of **billable** hours projected `FORECAST_WEEKS` ahead per department. Directional booking-pace aid, not a statistical model.

In [ ]:
# ============================================================
# SECTION 1: Configuration & helpers
# ------------------------------------------------------------
# WHAT: Imports, Spark session, run parameters, and two small
#       helpers used by every section below.
# WHY:  Centralising config means a single edit (e.g. capacity)
#       re-flows through all metrics. Mirrors the bronze notebook's
#       CONFIG block (tech_coe_allocation_bronze.ipynb, Section 1).
#
# PARAMETERS
#   SOURCE_TABLE            bronze output we read from.
#   TARGET_SCHEMA           where the 6 gold tables are written.
#   CAPACITY_HOURS_PER_WEEK denominator for utilization (40 = full time).
#   FORECAST_WEEKS          how many weeks Section 7 projects forward.
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType, IntegerType
)
import pandas as pd
import numpy as np

spark = SparkSession.builder.appName('tech_coe_allocations_gold').getOrCreate()

# ─── CONFIG ────────────────────────────────────────
SOURCE_TABLE  = 'arganoadw_oacuser_sh.oacuser.TECH_COE_ALLOCATIONS'
TARGET_SCHEMA = 'arganoadw_oacuser_sh.oacuser'

CAPACITY_HOURS_PER_WEEK = 40.0   # full-time weekly capacity per Arganaut
FORECAST_WEEKS          = 8      # weeks to project beyond the last scheduled week
# ───────────────────────────────────────────────

CAP = CAPACITY_HOURS_PER_WEEK    # short alias used inside the SQL below

def gold_table(name):
    # Build the fully-qualified 3-part table name for the external catalog.
    return f'{TARGET_SCHEMA}.{name}'

def write_gold(df, name):
    # Write one gold table. Same external-catalog (ADW) rules as the bronze
    # notebook's Section 5: NO .format('delta'), NO CREATE SCHEMA, and use
    # saveAsTable with overwrite + overwriteSchema so the table is rebuilt
    # cleanly each run (full refresh, not incremental).
    tbl = gold_table(name)
    (df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(tbl))
    print(f'✅ wrote {tbl}  ({df.count():,} rows)')
    return tbl

In [ ]:
# ============================================================
# SECTION 2: Load bronze fact & classify hour types
# ------------------------------------------------------------
# WHAT: Read the bronze table, tag every row with HOURS_TYPE
#       (BILLABLE / TIME_OFF / HOLIDAY), normalise YEAR_MONTH,
#       then materialise it.
# WHY:  Holidays and PTO are booked on the project line, so raw
#       SCHEDULED_HOURS mixes billable work with time off. Tagging
#       once here lets every downstream section treat them correctly.
#
# CLASSIFICATION (matches the raw spreadsheet text the bronze
# notebook split into PROJECT_CODE / PROJECT_NAME):
#   TIME_OFF = 'ARG* - Time Off Bookings'  (any ARG* code; match on name)
#   HOLIDAY  = 'Holiday - Holiday'
#   BILLABLE = everything else
#
# MONTH-STRADDLE FIX (re-derive YEAR_MONTH from WEEK_START_DATE):
#   A week that crosses a month-end (e.g. 2026-06-28 spans Jun 28-30 +
#   Jul 1-4) is split by the source into two columns sharing the same
#   WEEK_START_DATE but tagged different months. Because the week grain
#   includes YEAR_MONTH, that one calendar week would otherwise become
#   two person-week rows, each charged a full 40h capacity (80h total),
#   halving utilization. Re-deriving YEAR_MONTH from WEEK_START_DATE
#   assigns the week to its START-date month, collapsing the split back
#   to a single 40h week. (To use a different rule — e.g. ISO Thursday
#   or majority-of-days — change the date_format expression below.)
#
# NOTE ON localCheckpoint:
#   The Oracle ADW connector pushes GROUP BY aggregations down to the
#   source and cannot translate SUM(CASE WHEN HOURS_TYPE = ...) over the
#   derived column (scala.MatchError in GenericScanBuilder.pushAggregation).
#   localCheckpoint severs the lineage back to ADW so every aggregation in
#   Sections 3-6 computes in Spark instead of being pushed to ADW.
# ============================================================
base = (spark.table(SOURCE_TABLE)
        .withColumn('HOURS_TYPE', expr('''
            CASE
              WHEN lower(trim(PROJECT_NAME)) = 'time off bookings' THEN 'TIME_OFF'
              WHEN lower(trim(PROJECT_CODE)) = 'holiday'
                OR lower(trim(PROJECT_NAME)) = 'holiday' THEN 'HOLIDAY'
              ELSE 'BILLABLE'
            END
        '''))
        # one calendar week -> one month (see MONTH-STRADDLE FIX above)
        .withColumn('YEAR_MONTH', expr("date_format(WEEK_START_DATE, 'yyyy-MMM')")))

base = base.localCheckpoint(eager=True)   # materialise; see note above
base.createOrReplaceTempView('alloc')     # 'alloc' = the tagged fact, used everywhere below
total = base.count()

rng = base.selectExpr('min(WEEK_START_DATE) AS lo', 'max(WEEK_START_DATE) AS hi').first()
print(f'Source rows: {total:,}   Week range: {rng.lo} → {rng.hi}')

# Sanity check 0: confirm no WEEK_START_DATE maps to more than one month
# (should print 0 rows after the straddle fix).
print('\nStraddle weeks remaining (expect none):')
spark.sql('''
    SELECT WEEK_START_DATE, COUNT(DISTINCT YEAR_MONTH) AS MONTHS
    FROM alloc GROUP BY WEEK_START_DATE HAVING COUNT(DISTINCT YEAR_MONTH) > 1
''').show(truncate=False)

# Sanity check 1: how the hours split across the three types.
print('Hours by type:')
spark.sql('''
    SELECT HOURS_TYPE, ROUND(SUM(SCHEDULED_HOURS), 1) AS HOURS, COUNT(*) AS ROWS
    FROM alloc GROUP BY HOURS_TYPE ORDER BY HOURS DESC
''').show(truncate=False)

# Sanity check 2: list every non-billable line so the classification can be
# eyeballed (e.g. confirm all ARG* Time Off codes were caught).
print('Lines classified as Time Off / Holiday — verify these are correct:')
spark.sql('''
    SELECT DISTINCT HOURS_TYPE, PROJECT_CODE, PROJECT_NAME
    FROM alloc WHERE HOURS_TYPE <> 'BILLABLE'
    ORDER BY HOURS_TYPE, PROJECT_CODE
''').show(50, truncate=False)

In [ ]:
# ============================================================
# SECTION 3: Person-week utilization
# ------------------------------------------------------------
# WHAT: One row per Arganaut per week with their billable load,
#       time off, capacity, and utilization (gross + net).
# WHY:  The atomic 'who is busy / who is open' view. Feeds the
#       department roll-up in Section 4 and answers staffing
#       questions at the individual level.
#
# FORMULAS (CAP = 40):
#   BILLABLE_HOURS           = sum of BILLABLE scheduled hours
#   TOTAL_TIME_OFF_HOURS     = sum of TIME_OFF + HOLIDAY hours
#   AVAILABLE_CAPACITY_HOURS = CAP − time off
#   GROSS_UTILIZATION_PCT    = billable ÷ CAP × 100
#   NET_UTILIZATION_PCT      = billable ÷ available capacity × 100
#                              (NULLIF guards a fully-off week → NULL, not /0)
#   NONBILLABLE_HOURS        = max(CAP − all scheduled, 0)  (idle, sellable)
#   IS_OVER_ALLOCATED        = billable > available capacity (booked past
#                              what they can work after time off)
#   IS_ON_BENCH              = no billable work but capacity is free
# ============================================================
person_week = spark.sql(f'''
    SELECT DEPARTMENT, ARGANAUT, YEAR_MONTH, WEEK_START_DATE,
           ROUND(SUM(SCHEDULED_HOURS), 2)                                       AS SCHEDULED_HOURS,
           {CAP}                                                                AS CAPACITY_HOURS,
           ROUND(SUM(CASE WHEN HOURS_TYPE = 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END), 2) AS BILLABLE_HOURS,
           ROUND(SUM(CASE WHEN HOURS_TYPE = 'TIME_OFF' THEN SCHEDULED_HOURS ELSE 0 END), 2) AS TIME_OFF_HOURS,
           ROUND(SUM(CASE WHEN HOURS_TYPE = 'HOLIDAY'  THEN SCHEDULED_HOURS ELSE 0 END), 2) AS HOLIDAY_HOURS,
           ROUND(SUM(CASE WHEN HOURS_TYPE <> 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END), 2) AS TOTAL_TIME_OFF_HOURS,
           ROUND({CAP} - SUM(CASE WHEN HOURS_TYPE <> 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END), 2) AS AVAILABLE_CAPACITY_HOURS,
           ROUND(SUM(CASE WHEN HOURS_TYPE = 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END) / {CAP} * 100, 1) AS GROSS_UTILIZATION_PCT,
           ROUND(SUM(CASE WHEN HOURS_TYPE = 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END)
                 / NULLIF({CAP} - SUM(CASE WHEN HOURS_TYPE <> 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END), 0) * 100, 1) AS NET_UTILIZATION_PCT,
           ROUND(GREATEST({CAP} - SUM(SCHEDULED_HOURS), 0), 2)                  AS NONBILLABLE_HOURS,
           CASE WHEN SUM(CASE WHEN HOURS_TYPE = 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END)
                     > {CAP} - SUM(CASE WHEN HOURS_TYPE <> 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END)
                THEN 1 ELSE 0 END                                              AS IS_OVER_ALLOCATED,
           CASE WHEN SUM(CASE WHEN HOURS_TYPE = 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END) = 0
                AND {CAP} - SUM(CASE WHEN HOURS_TYPE <> 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END) > 0
                THEN 1 ELSE 0 END                                              AS IS_ON_BENCH
    FROM alloc
    GROUP BY DEPARTMENT, ARGANAUT, YEAR_MONTH, WEEK_START_DATE
''')
person_week.createOrReplaceTempView('person_week')
write_gold(person_week, 'TECH_COE_ALLOC_UTILIZATION_PERSON_WEEK')

In [ ]:
# ============================================================
# SECTION 4: Department-week summary  (the exec headline)
# ------------------------------------------------------------
# WHAT: Roll the person-week view up to one row per department
#       per week — the numbers leadership looks at first.
# WHY:  Shows, per department per week: how utilised the team is,
#       how much capacity is free to sell (bench), and how much
#       time off is removing capacity.
#
# FORMULAS:
#   ROSTER_HEADCOUNT          = distinct Arganauts active that week
#   TOTAL_CAPACITY_HOURS      = roster × CAP
#   AVAILABLE_CAPACITY_HOURS  = capacity − time off
#   GROSS_UTILIZATION_PCT     = billable ÷ capacity × 100
#   NET_UTILIZATION_PCT       = billable ÷ available capacity × 100
#   NONBILLABLE_HOURS (bench) = max(capacity − all scheduled, 0)
#
# The second query (pw_agg) counts over-allocated / bench people from
# Section 3 and is joined on; those are headcounts, not hour sums, so
# they can't be computed in the same GROUP BY.
# ============================================================
dept_week = spark.sql(f'''
    SELECT DEPARTMENT, YEAR_MONTH, WEEK_START_DATE,
           COUNT(DISTINCT ARGANAUT)                                            AS ROSTER_HEADCOUNT,
           COUNT(DISTINCT CASE WHEN HOURS_TYPE = 'BILLABLE' AND SCHEDULED_HOURS > 0 THEN ARGANAUT END) AS BILLABLE_HEADCOUNT,
           ROUND(SUM(CASE WHEN HOURS_TYPE = 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END), 2) AS BILLABLE_HOURS,
           ROUND(SUM(CASE WHEN HOURS_TYPE = 'TIME_OFF' THEN SCHEDULED_HOURS ELSE 0 END), 2) AS TIME_OFF_HOURS,
           ROUND(SUM(CASE WHEN HOURS_TYPE = 'HOLIDAY'  THEN SCHEDULED_HOURS ELSE 0 END), 2) AS HOLIDAY_HOURS,
           ROUND(SUM(CASE WHEN HOURS_TYPE <> 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END), 2) AS TOTAL_TIME_OFF_HOURS,
           ROUND(COUNT(DISTINCT ARGANAUT) * {CAP}, 2)                          AS TOTAL_CAPACITY_HOURS,
           ROUND(COUNT(DISTINCT ARGANAUT) * {CAP}
                 - SUM(CASE WHEN HOURS_TYPE <> 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END), 2) AS AVAILABLE_CAPACITY_HOURS,
           ROUND(SUM(CASE WHEN HOURS_TYPE = 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END)
                 / (COUNT(DISTINCT ARGANAUT) * {CAP}) * 100, 1)                AS GROSS_UTILIZATION_PCT,
           ROUND(SUM(CASE WHEN HOURS_TYPE = 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END)
                 / NULLIF(COUNT(DISTINCT ARGANAUT) * {CAP}
                          - SUM(CASE WHEN HOURS_TYPE <> 'BILLABLE' THEN SCHEDULED_HOURS ELSE 0 END), 0) * 100, 1) AS NET_UTILIZATION_PCT,
           ROUND(GREATEST(COUNT(DISTINCT ARGANAUT) * {CAP} - SUM(SCHEDULED_HOURS), 0), 2) AS NONBILLABLE_HOURS
    FROM alloc
    GROUP BY DEPARTMENT, YEAR_MONTH, WEEK_START_DATE
''')

# Headcount-based risk metrics, rolled up from the person-week view.
pw_agg = spark.sql('''
    SELECT DEPARTMENT, YEAR_MONTH, WEEK_START_DATE,
           SUM(IS_OVER_ALLOCATED) AS OVER_ALLOCATED_COUNT,
           SUM(IS_ON_BENCH)       AS BENCH_HEADCOUNT
    FROM person_week
    GROUP BY DEPARTMENT, YEAR_MONTH, WEEK_START_DATE
''')

dept_week = dept_week.join(pw_agg, ['DEPARTMENT', 'YEAR_MONTH', 'WEEK_START_DATE'], 'left')
dept_week.createOrReplaceTempView('dept_week')   # reused by the forecast (Section 7)
write_gold(dept_week, 'TECH_COE_ALLOC_DEPT_WEEK')

In [ ]:
# ============================================================
# SECTION 5: Project demand & key-person risk
# ------------------------------------------------------------
# WHAT: Two project-centric tables, BILLABLE lines only.
#       a. project_week — hours & headcount per project per week
#          (the demand / staffing curve over time).
#       b. project_risk — per-project totals + a key-person-risk flag.
# WHY:  Shows where the work is concentrated and which projects
#       depend on a single person.
#
# WHY 'WHERE HOURS_TYPE = BILLABLE': Time Off / Holiday are not real
#   projects; including them would pollute the top-projects ranking.
#
# RISK FLAG:
#   IS_KEY_PERSON_RISK = 1 when exactly one person is staffed on the
#   project across the whole horizon (bus-factor of one).
# ============================================================
project_week = spark.sql('''
    SELECT DEPARTMENT, PROJECT_CODE, PROJECT_NAME,
           YEAR_MONTH, WEEK_START_DATE,
           ROUND(SUM(SCHEDULED_HOURS), 2)                                 AS SCHEDULED_HOURS,
           COUNT(DISTINCT CASE WHEN SCHEDULED_HOURS > 0 THEN ARGANAUT END) AS HEADCOUNT
    FROM alloc
    WHERE HOURS_TYPE = 'BILLABLE'
    GROUP BY DEPARTMENT, PROJECT_CODE, PROJECT_NAME, YEAR_MONTH, WEEK_START_DATE
''')
write_gold(project_week, 'TECH_COE_ALLOC_PROJECT_WEEK')

project_risk = spark.sql('''
    SELECT PROJECT_CODE, PROJECT_NAME,
           ROUND(SUM(SCHEDULED_HOURS), 2)                                 AS TOTAL_SCHEDULED_HOURS,
           COUNT(DISTINCT CASE WHEN SCHEDULED_HOURS > 0 THEN ARGANAUT END) AS DISTINCT_PEOPLE,
           COUNT(DISTINCT CASE WHEN SCHEDULED_HOURS > 0 THEN WEEK_START_DATE END) AS ACTIVE_WEEKS,
           CASE WHEN COUNT(DISTINCT CASE WHEN SCHEDULED_HOURS > 0 THEN ARGANAUT END) = 1
                THEN 1 ELSE 0 END                                         AS IS_KEY_PERSON_RISK
    FROM alloc
    WHERE HOURS_TYPE = 'BILLABLE'
    GROUP BY PROJECT_CODE, PROJECT_NAME
    HAVING SUM(SCHEDULED_HOURS) > 0
''')
write_gold(project_risk, 'TECH_COE_ALLOC_PROJECT_RISK')

In [ ]:
# ============================================================
# SECTION 6: Detail fact (flexible slicing base)
# ------------------------------------------------------------
# WHAT: A near-passthrough of the tagged fact at full grain
#       (dept × arganaut × project × week) carrying HOURS_TYPE.
# WHY:  The aggregated tables above answer fixed questions; this
#       one lets OAC pivot freely — slice by Department AND Project,
#       and include or exclude time off via the HOURS_TYPE column.
#       Point ad-hoc / drill-down dashboards here.
# ============================================================
detail = spark.sql('''
    SELECT DEPARTMENT, ARGANAUT, PROJECT_CODE, PROJECT_NAME, HOURS_TYPE,
           YEAR_MONTH, WEEK_START_DATE, SCHEDULED_HOURS
    FROM alloc
''')
write_gold(detail, 'TECH_COE_ALLOC_DETAIL')

In [ ]:
# ============================================================
# SECTION 7: Billable demand forecast (per department)
# ------------------------------------------------------------
# WHAT: For each department, fit a straight-line trend to weekly
#       BILLABLE hours and project it FORECAST_WEEKS into the future,
#       alongside a 3-week moving average.
# WHY:  Gives leadership a directional read on booking pace — is
#       committed billable work trending up or down?
#
# METHOD (kept dependency-light — numpy only):
#   x = week index (0,1,2,…)         y = billable hours that week
#   slope, intercept = np.polyfit(x, y, 1)   # least-squares line
#   TREND_HOURS  = intercept + slope · x     (clipped at 0)
#   MA_3WK_HOURS = rolling 3-week mean of billable hours
#   Future weeks: ACTUAL_BILLABLE_HOURS is NULL; IS_FORECAST = 1.
#
# CAVEAT: the source is forward allocations that taper in far-future
#   weeks (the 'coverage cliff'), so the trend reflects current booking
#   pace — read the near-term weeks, not the long tail. Not a
#   statistical model; no seasonality or confidence interval.
#
# Computed in pandas (small, one row per dept-week) rather than Spark
# because np.polyfit / rolling are simplest there.
# ============================================================
from datetime import timedelta

pdf = dept_week.select('DEPARTMENT', 'WEEK_START_DATE', 'BILLABLE_HOURS').toPandas()
pdf['WEEK_START_DATE'] = pd.to_datetime(pdf['WEEK_START_DATE'])

rows = []
for dept, g in pdf.groupby('DEPARTMENT'):
    g = g.sort_values('WEEK_START_DATE').reset_index(drop=True)
    ma = g['BILLABLE_HOURS'].rolling(3, min_periods=1).mean()
    # week index in weeks since the department's first week
    x = ((g['WEEK_START_DATE'] - g['WEEK_START_DATE'].min()).dt.days / 7.0).values
    y = g['BILLABLE_HOURS'].astype(float).values
    if len(g) >= 2:
        slope, intercept = np.polyfit(x, y, 1)        # need >=2 points to fit a line
    else:
        slope, intercept = 0.0, (y[0] if len(y) else 0.0)
    # actual weeks (IS_FORECAST = 0)
    for i in range(len(g)):
        rows.append([dept, g['WEEK_START_DATE'][i].date(), round(float(y[i]), 2),
                     round(float(intercept + slope * x[i]), 2), round(float(ma[i]), 2), 0])
    # projected future weeks (IS_FORECAST = 1, no actual)
    last_week = g['WEEK_START_DATE'].max()
    last_x = float(x.max()) if len(x) else 0.0
    last_ma = float(g['BILLABLE_HOURS'].tail(3).mean())   # carry last MA forward
    for k in range(1, FORECAST_WEEKS + 1):
        fdate = (last_week + timedelta(weeks=k)).date()
        rows.append([dept, fdate, None,
                     round(float(intercept + slope * (last_x + k)), 2),
                     round(last_ma, 2), 1])

fc = pd.DataFrame(rows, columns=['DEPARTMENT', 'WEEK_START_DATE',
                                 'ACTUAL_BILLABLE_HOURS', 'TREND_HOURS',
                                 'MA_3WK_HOURS', 'IS_FORECAST'])
fc['WEEK_START_DATE'] = pd.to_datetime(fc['WEEK_START_DATE'])
for c in ['ACTUAL_BILLABLE_HOURS', 'TREND_HOURS', 'MA_3WK_HOURS']:
    fc[c] = fc[c].astype(float)
fc['TREND_HOURS'] = fc['TREND_HOURS'].clip(lower=0)   # hours can't be negative
fc['IS_FORECAST'] = fc['IS_FORECAST'].astype(int)

fc_schema = StructType([
    StructField('DEPARTMENT',            StringType(),  True),
    StructField('WEEK_START_DATE',       DateType(),    True),
    StructField('ACTUAL_BILLABLE_HOURS', DoubleType(),  True),
    StructField('TREND_HOURS',           DoubleType(),  True),
    StructField('MA_3WK_HOURS',          DoubleType(),  True),
    StructField('IS_FORECAST',           IntegerType(), True),
])
forecast = spark.createDataFrame(fc, schema=fc_schema)

# ACTUAL_BILLABLE_HOURS is None on forecast rows; astype(float) turned those
# into NaN. The ADW COPY_DATA load rejects NaN in a NUMBER column
# (ORA-20003 'reject limit reached'), so convert NaN -> SQL NULL first.
forecast = forecast.withColumn(
    'ACTUAL_BILLABLE_HOURS',
    expr('CASE WHEN isnan(ACTUAL_BILLABLE_HOURS) THEN NULL ELSE ACTUAL_BILLABLE_HOURS END'))
write_gold(forecast, 'TECH_COE_ALLOC_DEPT_FORECAST')

In [ ]:
# ============================================================
# SECTION 8: Verify / preview
# ------------------------------------------------------------
# WHAT: Read back the written tables and print four leadership
#       views as a smoke test (mirrors the bronze notebook's
#       Section 6 verification step).
# WHY:  Confirms the tables landed and the numbers look sane
#       before anyone points a dashboard at them. Read-only.
#
# Checks: (1) dept utilization gross vs net, (2) company coverage
# cliff by week, (3) top billable projects + risk, (4) the forecast.
# ============================================================
print('── Department utilization (gross vs net of time off), recent weeks ──')
spark.sql(f'''
    SELECT DEPARTMENT, WEEK_START_DATE, ROSTER_HEADCOUNT,
           GROSS_UTILIZATION_PCT, NET_UTILIZATION_PCT,
           BILLABLE_HOURS, TOTAL_TIME_OFF_HOURS, NONBILLABLE_HOURS,
           OVER_ALLOCATED_COUNT
    FROM {gold_table('TECH_COE_ALLOC_DEPT_WEEK')}
    ORDER BY WEEK_START_DATE DESC, DEPARTMENT
    LIMIT 40
''').show(40, truncate=False)

print('── Coverage cliff: company-wide utilization by week ──')
spark.sql(f'''
    SELECT WEEK_START_DATE,
           ROUND(SUM(BILLABLE_HOURS) / SUM(TOTAL_CAPACITY_HOURS) * 100, 1)     AS GROSS_UTILIZATION_PCT,
           ROUND(SUM(BILLABLE_HOURS) / SUM(AVAILABLE_CAPACITY_HOURS) * 100, 1) AS NET_UTILIZATION_PCT,
           ROUND(SUM(NONBILLABLE_HOURS), 0)                                    AS BENCH_HOURS,
           ROUND(SUM(TOTAL_TIME_OFF_HOURS), 0)                                 AS TIME_OFF_HOURS
    FROM {gold_table('TECH_COE_ALLOC_DEPT_WEEK')}
    GROUP BY WEEK_START_DATE
    ORDER BY WEEK_START_DATE
''').show(60, truncate=False)

print('── Top 15 billable projects by scheduled hours (key-person risk flagged) ──')
spark.sql(f'''
    SELECT PROJECT_CODE, PROJECT_NAME,
           TOTAL_SCHEDULED_HOURS, DISTINCT_PEOPLE, IS_KEY_PERSON_RISK
    FROM {gold_table('TECH_COE_ALLOC_PROJECT_RISK')}
    ORDER BY TOTAL_SCHEDULED_HOURS DESC
    LIMIT 15
''').show(15, truncate=False)

print('── Billable demand forecast (future weeks) ──')
spark.sql(f'''
    SELECT DEPARTMENT, WEEK_START_DATE, TREND_HOURS, MA_3WK_HOURS
    FROM {gold_table('TECH_COE_ALLOC_DEPT_FORECAST')}
    WHERE IS_FORECAST = 1
    ORDER BY DEPARTMENT, WEEK_START_DATE
    LIMIT 40
''').show(40, truncate=False)